# 2일차 팀 프로젝트: 보안 문서 하이브리드 RAG (노트북 버전)

> `notebooks/day2_security_rag.py`(스크립트 버전)와 **독립적으로 공존하는 노트북 버전**입니다. 두 구현은 서로 참조하지 않으며 각자 실행됩니다.

## 프로젝트 목표
1. 보안 표준/가이드 PDF를 Qdrant Cloud에 저장
2. Parent Document Retriever 패턴 + 하이브리드 검색(BM25 + 벡터 RRF) 적용
3. 카테고리 메타데이터 필터링 및 구조화된 답변의 RAG 시스템 구현

## 팀원 이식 안내
`[이식용]` 표시가 붙은 셀 3개는 **그 셀만 복사해도 동작**하도록 작성했습니다.
각 셀 상단 주석에 전제 변수·변수명 매핑·반환값이 명시되어 있습니다.

| 셀 | 기능 | 요구 변수 | 제공 |
|---|---|---|---|
| [이식용 A] | 카테고리 메타데이터 + Qdrant payload 필터 | `docs` (필수), `client`·컬렉션명 (선택) | `category` 메타데이터, `build_qdrant_filter()`, `ensure_payload_indexes()` |
| [이식용 B] | BM25 + 벡터 RRF 하이브리드 | `child_docs`, `vectorstore`, `parent_docstore` | `tokenize()`, `bm25_index`, `hybrid_retriever` |
| [이식용 C] | Parent retriever k 튜닝 | `vectorstore`, `parent_docstore` | `ParentDocumentRetriever`, `parent_retriever` |

## 0. 환경 변수 설정

In [ ]:
# 필요 패키지 (pyproject.toml에 모두 반영되어 있음):
#   langchain, langchain-openai, langchain-qdrant, langchain-text-splitters,
#   pymupdf, python-dotenv, qdrant-client, ipython
#   rank-bm25   : BM25 하이브리드 검색용
#   kiwipiepy   : 한국어 형태소 토크나이저 (선택 — 없으면 정규식+불용어로 폴백)
import os
from dotenv import load_dotenv

load_dotenv()

# API Key 확인
if os.environ.get("OPENAI_API_KEY"):
    print("✓ OpenAI API Key가 설정되었습니다.")
else:
    print("✗ OpenAI API Key가 없습니다.")

# Qdrant Cloud 설정 확인
if os.environ.get("QDRANT_URL") and os.environ.get("QDRANT_API_KEY"):
    print("✓ Qdrant Cloud 설정이 완료되었습니다.")
else:
    print("✗ Qdrant Cloud 설정이 필요합니다.")
    print("  .env 파일에 QDRANT_URL과 QDRANT_API_KEY를 추가하세요.")

## 0-1. 경로·파라미터 설정

**환경이 다르면 이 셀의 변수만 수정하면 됩니다.**

In [ ]:
from pathlib import Path

# PDF 폴더 경로 — 환경에 맞게 이 변수 하나만 수정
#   이 저장소 기준        : ../datasets/보안 pdf모음
#   팀원 노트북(rag-system): Path("../datasets/보안 취약점 PDF")
PDF_DIR = Path("../datasets/보안 pdf모음")

# 청킹 설정 — 팀 합의값 900/150
# 참고: 이전 값 600/100은 실험 근거 없이 정한 휴리스틱이었음.
#       조항 단위가 긴 보안 표준 문서 특성상 900/150으로 통일.
CHUNK_SIZE = 900
CHUNK_OVERLAP = 150

# Qdrant 컬렉션 이름
COLLECTION_NAME = "security_docs"

print(f"PDF_DIR = {PDF_DIR}")
print(f"CHUNK_SIZE / CHUNK_OVERLAP = {CHUNK_SIZE} / {CHUNK_OVERLAP}")
print(f"COLLECTION_NAME = {COLLECTION_NAME}")

## 1. PDF 문서 로딩 (페이지 단위 Parent 문서)

In [ ]:
from langchain_core.documents import Document
import fitz

pdf_files = sorted(PDF_DIR.glob("*.pdf"))

print(f"로딩할 PDF 파일 ({len(pdf_files)}개):")
for f in pdf_files:
    print(f"  - {f.name}")

docs = []

for pdf_path in pdf_files:
    doc = fitz.open(pdf_path)
    file_name = pdf_path.name

    # 페이지 단위로 Document 생성 (Parent Document)
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text("text", sort=True)

        # 빈 페이지는 스킵
        if len(text.strip()) < 10:
            continue

        docs.append(
            Document(
                page_content=text,
                metadata={
                    "source": file_name,
                    "page": page_num + 1,
                    # 파일이 여러 개이므로 파일명을 포함해 고유한 parent_id 생성
                    "parent_id": f"{pdf_path.stem}_page_{page_num + 1}"
                }
            )
        )

    doc.close()

print(f"\n총 {len(docs)}개의 페이지(Parent Document) 로드 완료")
print(f"평균 페이지 길이: {sum(len(d.page_content) for d in docs) / len(docs):.0f}자")

## 1-1. [이식용] (a) 문서 카테고리 메타데이터 + Qdrant payload 필터

**이 셀만 복사하면 동작합니다.** Parent 문서에 `category` 메타데이터를 부여하고,
Qdrant 필터 생성 함수·payload 인덱스 생성 함수를 정의합니다.
Child chunk 생성 **전에** 실행하면 category가 chunk에 자동 상속됩니다.

In [ ]:
# [이식용] (a) 카테고리 메타데이터 부여 + Qdrant payload 필터링
# ─────────────────────────────────────────────────────────────
# 전제 변수:
#   docs            : list[Document]   (필수 — Parent 문서 리스트)
#   child_docs      : list[Document]   (선택 — 이미 만들었으면 함께 태깅됨)
#   client          : QdrantClient     (선택 — 있으면 payload 인덱스 즉시 생성)
#   COLLECTION_NAME 또는 collection_name : str (선택 — client와 함께 사용)
# 제공:
#   각 문서 metadata["category"] 채움
#   build_qdrant_filter(category=None, source=None) → qdrant Filter | None
#   ensure_payload_indexes(client, collection_name)  → payload 인덱스 생성(멱등)
# 변수명 매핑: 팀원 노트북에서 Parent 리스트 이름이 docs가 아니면
#   맨 아래 assign_categories(docs)의 docs만 그 이름으로 바꾸면 됩니다.
# 주의: Qdrant Cloud는 인덱스 없는 필드로 필터링할 수 없으므로,
#   업로드 후 ensure_payload_indexes()가 반드시 한 번 호출되어야 합니다.
# ─────────────────────────────────────────────────────────────
from collections import Counter
from qdrant_client.http import models as qmodels

# 파일명 → 카테고리 (새 PDF를 추가하면 여기에 등록, 없으면 "기타")
CATEGORY_MAP = {
    "CNA_Rules_v4.1.0.pdf": "CVE 관리",
    "cvss-v40-specification.pdf": "취약점 평가",
    "Federal_Government_Cybersecurity_Incident_and_Vulnerability_Response_Playbooks_508C.pdf": "사고 대응",
    "Key-Details-Phrasing.pdf": "CVE 작성 가이드",
    "NIST.SP.800-126r4.pdf": "SCAP 표준",
    "NIST.SP.800-216.pdf": "취약점 공개",
}

def assign_categories(documents):
    """source 파일명 기준으로 metadata["category"]를 채운다."""
    for d in documents:
        d.metadata["category"] = CATEGORY_MAP.get(d.metadata.get("source", ""), "기타")

def build_qdrant_filter(category=None, source=None):
    """category/source 조건의 Qdrant Filter 생성. 조건이 없으면 None.
    langchain-qdrant는 metadata를 payload["metadata"]에 저장하므로 key가 metadata.* 입니다."""
    conditions = []
    if category:
        conditions.append(qmodels.FieldCondition(
            key="metadata.category", match=qmodels.MatchValue(value=category)))
    if source:
        conditions.append(qmodels.FieldCondition(
            key="metadata.source", match=qmodels.MatchValue(value=source)))
    return qmodels.Filter(must=conditions) if conditions else None

def ensure_payload_indexes(client, collection_name):
    """필터링 대상 필드의 payload 인덱스 생성 (이미 있으면 그대로 유지)."""
    for field in ("metadata.category", "metadata.source"):
        client.create_payload_index(
            collection_name=collection_name, field_name=field, field_schema="keyword")
    print(f"payload 인덱스 확인 완료: metadata.category, metadata.source")

# ── 실행 ──
assign_categories(docs)
if "child_docs" in globals():
    assign_categories(child_docs)

print("카테고리 분포:", dict(Counter(d.metadata["category"] for d in docs)))

# client가 이미 있으면 인덱스도 바로 생성 (없으면 업로드 셀에서 호출)
_cn = globals().get("collection_name") or globals().get("COLLECTION_NAME")
if "client" in globals() and _cn:
    ensure_payload_indexes(client, _cn)

## 2. Child chunk 생성 (CHUNK_SIZE=900, CHUNK_OVERLAP=150)

팀 합의값 900/150을 사용합니다. (이전 600/100은 실험 근거가 없던 휴리스틱)

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

child_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP
)

# Parent를 Child chunk로 분할 — metadata는 통째로 복사해 category 등도 상속
child_docs = []

for parent_doc in docs:
    for chunk in child_splitter.split_text(parent_doc.page_content):
        child_docs.append(
            Document(page_content=chunk, metadata=dict(parent_doc.metadata))
        )

print(f"생성된 통계:")
print(f"  - Parent 문서 수: {len(docs)}")
print(f"  - Child chunk 수: {len(child_docs)}")
print(f"  - 평균 chunk/page: {len(child_docs) / len(docs):.1f}")

## 3. Qdrant Cloud 연결 및 Child chunk 업로드

In [ ]:
from langchain_qdrant import QdrantVectorStore
from langchain_openai import OpenAIEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from uuid import uuid4

client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY")
)

print("Qdrant Cloud에 연결되었습니다.")

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

collection_name = COLLECTION_NAME

# 컬렉션이 이미 있을 때 삭제 후 새로 만들지 여부
# (청킹 설정을 바꾼 경우에만 True로 두고 다시 업로드)
RECREATE_COLLECTION = False

collections = client.get_collections().collections
existing_collection = any(col.name == collection_name for col in collections)

need_upload = False

if existing_collection and RECREATE_COLLECTION:
    print(f"기존 컬렉션 '{collection_name}' 삭제 중...")
    client.delete_collection(collection_name=collection_name)
    existing_collection = False

if not existing_collection:
    client.create_collection(
        collection_name=collection_name,
        vectors_config=VectorParams(size=3072, distance=Distance.COSINE)
    )
    need_upload = True
    print(f"컬렉션 '{collection_name}' 생성 완료")
else:
    print(f"기존 컬렉션 '{collection_name}'을 사용합니다.")

# [이식용 A]에서 정의한 payload 인덱스 생성 (필터 검색에 필수)
ensure_payload_indexes(client, collection_name)

vectorstore = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

if need_upload:
    uuids = [str(uuid4()) for _ in range(len(child_docs))]
    vectorstore.add_documents(documents=child_docs, ids=uuids)
    print(f"\n{len(child_docs)}개의 Child chunk가 Qdrant Cloud에 추가되었습니다.")
else:
    print(f"\n기존 데이터를 사용합니다. (현재 {client.count(collection_name).count}개 포인트)")

## 4. Parent Document 저장 (Docstore)

In [ ]:
# Parent 문서를 dict에 저장 (parent_id를 키로 사용)
parent_docstore = {}

for parent_doc in docs:
    parent_docstore[parent_doc.metadata["parent_id"]] = parent_doc

print(f"Docstore에 {len(parent_docstore)}개의 Parent 문서 저장 완료")

## 5. [이식용] (c) Parent Document Retriever — k 튜닝

**이 셀만 복사하면 동작합니다.** parent k=4, child 후보는 k×3 검색 후 중복 제거합니다.
`vectorstore.similarity_search()` 인터페이스만 사용하므로 임베딩/로더 구현과 무관하게 동작합니다.

In [ ]:
# [이식용] (c) Parent Document Retriever (parent k=4, child k*3 + 중복 제거)
# ─────────────────────────────────────────────────────────────
# 전제 변수:
#   vectorstore     : similarity_search(query, k, [filter=...]) 인터페이스만 사용
#                     (langchain VectorStore 호환이면 무엇이든 가능 — 임베딩 구현 무관)
#   parent_docstore : dict[str, Document]  (parent_id → Parent 문서)
#   * child chunk의 metadata["parent_id"]가 parent_docstore의 키와 일치해야 함
# 선택 전제:
#   build_qdrant_filter — [이식용 A] 셀을 먼저 실행했으면 category/source 필터 사용 가능
#                         (없으면 필터 없이 동작)
# 제공:
#   ParentDocumentRetriever 클래스, parent_retriever 인스턴스
#   parent_retriever.invoke(query, category=None, source=None) → list[Document] (Parent k개)
#   parent_retriever.get_child_chunks(query, k=3, ...)         → list[Document] (비교용 child)
# 변수명 매핑: 팀원 노트북에 같은 이름의 ParentDocumentRetriever 클래스가 이미 있으면
#   이 클래스로 교체하거나, 클래스명을 ParentDocumentRetrieverV2로 바꿔 병행 사용하세요.
# ─────────────────────────────────────────────────────────────
from typing import List, Optional
from langchain_core.documents import Document

class ParentDocumentRetriever:
    """child chunk 검색 → parent_id 추출 → Parent 문서 반환"""

    def __init__(self, vectorstore, parent_docstore, k: int = 4, child_multiplier: int = 3):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.k = k                              # 최종 반환할 Parent 수
        self.child_multiplier = child_multiplier  # child 후보 = k * multiplier

    @staticmethod
    def _make_filter(category=None, source=None):
        # [이식용 A]가 실행된 경우에만 필터 생성, 아니면 None
        if "build_qdrant_filter" in globals():
            return build_qdrant_filter(category=category, source=source)
        return None

    def _search(self, query, k, category=None, source=None):
        flt = self._make_filter(category, source)
        kwargs = {"filter": flt} if flt is not None else {}
        return self.vectorstore.similarity_search(query, k=k, **kwargs)

    def invoke(self, query: str, category: Optional[str] = None,
               source: Optional[str] = None) -> List[Document]:
        # 1. child 후보를 넉넉히 검색 (같은 parent 중복 대비)
        child_results = self._search(query, self.k * self.child_multiplier, category, source)

        # 2. parent_id 추출 (등장 순서 유지 + 중복 제거)
        parent_ids = []
        for doc in child_results:
            pid = doc.metadata.get("parent_id")
            if pid and pid not in parent_ids:
                parent_ids.append(pid)
                if len(parent_ids) >= self.k:
                    break

        # 3. Docstore에서 Parent 문서 반환
        return [self.parent_docstore[pid] for pid in parent_ids
                if pid in self.parent_docstore]

    def get_child_chunks(self, query: str, k: int = 3,
                         category: Optional[str] = None,
                         source: Optional[str] = None) -> List[Document]:
        """비교용: child chunk 직접 반환"""
        return self._search(query, k, category, source)

parent_retriever = ParentDocumentRetriever(vectorstore, parent_docstore, k=4)
print("✓ Parent Document Retriever 생성 완료 (parent k=4, child k*3 + 중복 제거)")

## 5-1. [이식용] (b) 하이브리드 검색 — BM25 + 벡터, RRF 융합

**이 셀만 복사하면 동작합니다.** 토크나이저는 kiwipiepy 형태소 분석(조사·어미 제거)을
우선 사용하고, 미설치 시 정규식 + 불용어 폴백으로 동작합니다.

In [ ]:
# [이식용] (b) BM25 인덱스 구축 + 벡터 검색과 RRF 융합
# ─────────────────────────────────────────────────────────────
# 전제 변수:
#   child_docs      : list[Document]  (BM25 인덱싱 대상 chunk 리스트)
#                     * 팀원 노트북에서 chunk 리스트 이름이 다르면(예: chunks)
#                       아래 build 부분의 child_docs만 그 이름으로 바꾸세요.
#   vectorstore     : similarity_search(query, k, [filter=...]) 인터페이스만 사용
#   parent_docstore : dict[str, Document]
# 필요 패키지: rank-bm25 (pip install rank-bm25), kiwipiepy(선택)
# 선택 전제: build_qdrant_filter — [이식용 A] 실행 시 category 필터 사용 가능
# 제공:
#   tokenize(), bm25_index, HybridParentRetriever 클래스, hybrid_retriever 인스턴스
#   hybrid_retriever.invoke(query, category=None) → list[Document] (Parent k개)
# ─────────────────────────────────────────────────────────────
from rank_bm25 import BM25Okapi
import re

# ── 토크나이저: kiwipiepy 형태소 분석 → 미설치 시 정규식+불용어 폴백 ──
try:
    from kiwipiepy import Kiwi
    _kiwi = Kiwi()
    # 내용어만 유지: 명사(NN*/NR/NP), 동사·형용사 어간(VV/VA), 어근(XR),
    # 외국어(SL), 숫자(SN), 한자(SH) — 조사(J*)·어미(E*)는 자동 제거됨
    _KEEP_TAGS = ("NNG", "NNP", "NNB", "NR", "NP", "VV", "VA", "XR", "SL", "SN", "SH")

    def tokenize(text: str) -> list:
        return [t.form.lower() for t in _kiwi.tokenize(text) if t.tag in _KEEP_TAGS]

    print("토크나이저: kiwipiepy 형태소 분석 (조사/어미 제거)")
except ImportError:
    # 폴백: 영문/숫자/한글 정규식 + 최소 불용어 + 1글자 토큰 제거
    _STOPWORDS = {
        "및", "등", "것", "수", "때", "그", "이", "저", "은", "는", "을", "를", "의",
        "에", "에서", "으로", "로", "와", "과", "도", "만", "하다", "있다", "되다",
        "the", "a", "an", "of", "to", "in", "for", "and", "or", "is", "are", "be",
        "on", "by", "with", "as", "at", "that", "this", "it", "from", "was", "were",
    }

    def tokenize(text: str) -> list:
        toks = re.findall(r"[a-z0-9]+|[가-힣]+", text.lower())
        return [t for t in toks if t not in _STOPWORDS and len(t) > 1]

    print("토크나이저: 정규식 + 불용어 폴백 (kiwipiepy 미설치)")

# ── BM25 인덱스 구축 (로컬, 무료) ──
bm25_index = BM25Okapi([tokenize(d.page_content) for d in child_docs])

class HybridParentRetriever:
    """BM25(키워드)와 벡터 검색의 parent 순위를 RRF로 융합해 Parent 문서 반환

    RRF(Reciprocal Rank Fusion): score = Σ 1/(rrf_k + rank)
    CVE-2024-1234, SCAP 같은 정확한 식별자는 BM25가, 의미 기반 질문(한국어 질문
    ↔ 영어 문서)은 벡터 검색이 강함 → 상호 보완."""

    def __init__(self, vectorstore, parent_docstore, child_docs, bm25_index,
                 k: int = 4, candidate_k: int = 12, rrf_k: int = 60):
        self.vectorstore = vectorstore
        self.parent_docstore = parent_docstore
        self.child_docs = child_docs
        self.bm25_index = bm25_index
        self.k = k                      # 최종 반환 Parent 수
        self.candidate_k = candidate_k  # 검색기별 child 후보 수
        self.rrf_k = rrf_k              # RRF 완충 상수

    def _vector_parent_ranking(self, query, category=None):
        flt = build_qdrant_filter(category=category) \
            if ("build_qdrant_filter" in globals() and category) else None
        kwargs = {"filter": flt} if flt is not None else {}
        results = self.vectorstore.similarity_search(query, k=self.candidate_k, **kwargs)
        ranking = []
        for doc in results:
            pid = doc.metadata.get("parent_id")
            if pid and pid not in ranking:
                ranking.append(pid)
        return ranking

    def _bm25_parent_ranking(self, query, category=None):
        scores = self.bm25_index.get_scores(tokenize(query))
        order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        ranking = []
        for i in order:
            if scores[i] <= 0:
                break
            doc = self.child_docs[i]
            if category and doc.metadata.get("category") != category:
                continue
            pid = doc.metadata.get("parent_id")
            if pid and pid not in ranking:
                ranking.append(pid)
            if len(ranking) >= self.candidate_k:
                break
        return ranking

    def invoke(self, query, category=None):
        vector_rank = self._vector_parent_ranking(query, category)
        bm25_rank = self._bm25_parent_ranking(query, category)

        rrf_scores = {}
        for ranking in (vector_rank, bm25_rank):
            for rank, pid in enumerate(ranking, start=1):
                rrf_scores[pid] = rrf_scores.get(pid, 0) + 1 / (self.rrf_k + rank)

        top_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)[:self.k]
        return [self.parent_docstore[pid] for pid in top_ids
                if pid in self.parent_docstore]

hybrid_retriever = HybridParentRetriever(
    vectorstore=vectorstore,
    parent_docstore=parent_docstore,
    child_docs=child_docs,
    bm25_index=bm25_index,
    k=4
)
print("✓ Hybrid Parent Retriever 생성 완료 (BM25 + 벡터, RRF 융합)")

## 6. 검색 테스트 (벡터 vs 하이브리드 vs 카테고리 필터)

In [ ]:
query = "CVSS v4.0에서 취약점 심각도 등급(Critical, High, Medium, Low)은 어떤 점수 범위로 나뉘나요?"

def print_parents(results):
    for i, r in enumerate(results, start=1):
        print(f"\n  [{i}] {r.metadata.get('source', '?')} "
              f"(p.{r.metadata.get('page', '?')}, 카테고리: {r.metadata.get('category', '?')})")
        print(f"      미리보기: {r.page_content[:150].strip()}...")

print(f"검색 쿼리: {query}")

print("\n" + "=" * 80)
print("[1] Parent 검색 (벡터만)")
print_parents(parent_retriever.invoke(query))

print("\n" + "=" * 80)
print("[2] Parent 검색 (하이브리드: BM25 + 벡터, RRF)")
print_parents(hybrid_retriever.invoke(query))

print("\n" + "=" * 80)
print("[3] 카테고리 필터 검색 (category='취약점 평가')")
print_parents(hybrid_retriever.invoke(query, category="취약점 평가"))

## 7. RAG 시스템 구현 (하이브리드 검색 + 구조화된 답변 형식)

In [ ]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from IPython.display import Markdown, display

llm = init_chat_model("gpt-5.4-mini")

# 답변 형식을 구체적으로 지정한 프롬프트
template = """
당신은 사이버 보안 표준 및 취약점 관리 전문가입니다.
CVE/CNA 규칙, CVSS 취약점 평가, NIST 보안 표준(SCAP, 취약점 공개 가이드라인),
사이버 사고 대응 플레이북에 대한 깊은 지식을 갖추고 있습니다.

주어진 [참고 정보]만을 근거로 답변하세요.
참고 문서가 영어라도 한국어로 알기 쉽게 설명하되, 핵심 용어는 원문(영어)을 병기하세요.
참고 정보에 없는 내용은 추측하지 말고 "제공된 문서에서 찾을 수 없습니다"라고 답하세요.

반드시 아래 형식으로 답변하세요:

### 핵심 답변
(질문에 대한 직접적인 답을 2~3문장으로 요약)

### 상세 설명
(근거가 되는 내용을 목록이나 표로 구조화하여 설명.
 항목별 기준·수치·단계가 있으면 표로 정리)

### 출처
(참고한 문서명과 페이지 번호를 목록으로: - 문서명, p.페이지)

[참고 정보]
{context}

[질문]
{question}
"""

prompt_template = PromptTemplate(
    input_variables=["context", "question"],
    template=template
)

def rag_with_parent_retriever(question: str, category: str = None,
                              use_hybrid: bool = True) -> str:
    """Parent Document RAG — use_hybrid로 하이브리드/벡터 선택, category로 범위 한정"""
    retriever = hybrid_retriever if use_hybrid else parent_retriever
    retrieved_docs = retriever.invoke(question, category=category)

    context_parts = []
    for doc in retrieved_docs:
        context_parts.append(
            f"[출처: {doc.metadata.get('source', '?')}, "
            f"페이지: {doc.metadata.get('page', '?')}, "
            f"카테고리: {doc.metadata.get('category', '?')}]\n{doc.page_content}"
        )
    context = "\n\n---\n\n".join(context_parts)

    formatted_prompt = prompt_template.format(context=context, question=question)
    return llm.invoke(formatted_prompt).content

print("✓ RAG 시스템 준비 완료")

## 8. RAG 시스템 테스트

In [ ]:
questions = [
    "CVSS v4.0의 Base Metric에는 어떤 항목들이 있고 각각 무엇을 평가하나요?",
    "CNA가 CVE ID를 할당할 수 있는 취약점의 기준은 무엇인가요?",
    "연방정부 사이버 사고 대응 플레이북에서 정의하는 사고 대응(Incident Response) 단계는 무엇인가요?"
]

for q in questions:
    print(f"\n{'=' * 80}")
    print(f"질문: {q}")
    print(f"{'=' * 80}\n")
    display(Markdown(rag_with_parent_retriever(q)))

## 이식용 셀 요약

| 셀 | 요구 변수 | 제공(반환) |
|---|---|---|
| **[이식용 A]** 카테고리 + payload 필터 | `docs` (필수) / `child_docs`, `client`+컬렉션명 (선택) | 문서에 `category` 메타데이터, `build_qdrant_filter()`, `ensure_payload_indexes()` |
| **[이식용 B]** BM25 + RRF 하이브리드 | `child_docs`, `vectorstore`, `parent_docstore` | `tokenize()`, `bm25_index`, `hybrid_retriever.invoke(query, category=None)` |
| **[이식용 C]** retriever k 튜닝 | `vectorstore`, `parent_docstore` | `ParentDocumentRetriever` 클래스, `parent_retriever.invoke(query, category=None, source=None)` |

**이식 순서 권장**: (A) → 팀원 노트북 섹션 1과 2 사이 (category 상속을 위해 chunk 생성 전),
(C) → 섹션 4 대체 또는 병행, (B) → 섹션 4와 5 사이.
A 없이 B/C만 복사해도 동작합니다 (필터 기능만 비활성).

## 프로젝트 점검 체크리스트

- [x] PDF 문서 선정 및 로딩 (보안 PDF 6종, 경로는 상단 `PDF_DIR`로 일원화)
- [x] Child chunk 생성 (900/150 — 팀 합의값)
- [x] Qdrant Cloud 저장 + payload 인덱스
- [x] Parent Document Retriever (k=4, child k*3 + 중복 제거)
- [x] BM25 토크나이저 형태소 분석 적용 (kiwipiepy, 폴백 포함)
- [x] 하이브리드 검색 (BM25 + 벡터 RRF)
- [x] RAG 구조화 답변 (핵심 답변/상세 설명/출처)
- [x] 3개 이상 질문 테스트